# Tranche 1 — executable D2/D3 evidence

Run cells in order using a Python 3 kernel from this repository or its `notebooks/` directory. Only the Python standard library is used. Rust 1.84.1, Stellar CLI 22.8.2, cargo-llvm-cov 0.6.16 and the local-node binaries must already be installed; see [local RPC setup](../docs/local-rpc.md). This notebook does not install tools or reset existing node data.

**Recording:** run P0–P4 before filming; keep the node alive. On camera, run D2-A through D3-C. Full checks and deployment take longer than the two-minute D2 segment. A warm build is labeled as such. All transactions here use isolated standalone funds and fixture assets. No external network transaction is signed.

**Pricing:** Reflector supplies the NAV mark; the three-observation Soroswap TWAP is the circuit breaker. This reverses the original award's source ordering. Local tests use a controlled Reflector ABI fixture and actual Soroswap pair WASM, not deployed Reflector v6. [D3 acceptance](../docs/d3-acceptance.md) records outstanding full-universe/source/liquidity gates. This notebook does not close them.

Outputs are generated by execution, with dates, revision and artifact hashes. Do not present historical outputs as a fresh run. Logs, node state and new receipt summaries stay in ignored `artifacts/local/` and `.tooling/` directories.

## P0 — repository, pinned tools and evidence helpers

Use the optional repository-local tool installation when present, otherwise the tools on PATH. The notebook records the Git revision and working-tree status; changes after the recorded commit remain visible.

In [1]:
import hashlib
import json
import os
from pathlib import Path
import shlex
import socket
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from decimal import Decimal
import urllib.request

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/local-smoke.py").is_file())
ENV = os.environ.copy()
if (ROOT / ".tooling/cargo/bin/cargo").exists():
    ENV.update(RUSTUP_HOME=str(ROOT / ".tooling/rustup"),
               CARGO_HOME=str(ROOT / ".tooling/cargo"))
    ENV["PATH"] = f"{ROOT}/.tooling/bin:{ROOT}/.tooling/cargo/bin:" + ENV["PATH"]
OUT = ROOT / "artifacts/local/tranche-1-demo"
OUT.mkdir(parents=True, exist_ok=True)
RUN_AT = datetime.now(timezone.utc).isoformat()
RPC = "http://127.0.0.1:8003"
PASSPHRASE = "Standalone Network ; September 2026"

def run(args, label, tail=24):
    print("$", shlex.join(["python3" if a == sys.executable else a for a in args]), flush=True)
    started = time.monotonic()
    result = subprocess.run(args, cwd=ROOT, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (OUT / (label + ".log")).write_text(result.stdout)
    print("\n".join(result.stdout.replace(str(ROOT), ".").splitlines()[-tail:]))
    print(f"Exit {result.returncode}; {time.monotonic() - started:.1f}s; log: artifacts/local/tranche-1-demo/{label}.log")
    result.check_returncode()
    return result.stdout

def rpc(method, **params):
    request = urllib.request.Request(RPC, data=json.dumps({
        "jsonrpc": "2.0", "id": 1, "method": method, **({"params": params} if params else {})
    }).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=20) as response:
        payload = json.load(response)
    assert "error" not in payload, payload
    return payload["result"]

def invoke(contract, method, args=(), send="no"):
    # Every invocation rechecks the network before using a local fixture identity.
    assert rpc("getNetwork")["passphrase"] == PASSPHRASE
    result = subprocess.run([
        "stellar", "contract", "invoke", "--network", "local",
        "--source", "local-root", "--id", contract, "--send=" + send,
        "--", method, *args
    ], cwd=ROOT, env=ENV, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr)
    return json.loads(result.stdout) if result.stdout.strip() else None

def test(package, name):
    output = run(["cargo", "test", "--locked", "-p", package,
                  "tests::" + name, "--", "--exact", "--nocapture"], name, tail=16)
    assert "1 passed; 0 failed" in output, "The exact named test did not run"

print("Run UTC:", RUN_AT)
REVISION = run(["git", "rev-parse", "HEAD"], "revision").strip()
run(["git", "status", "--short"], "worktree")
assert "rustc 1.84.1 " in run(["rustc", "--version"], "rust")
assert "stellar 22.8.2 " in run(["stellar", "--version"], "stellar")
assert "0.6.16" in run(["cargo", "llvm-cov", "--version"], "coverage-tool")

Run UTC: 2026-09-19T13:33:11.778333+00:00
$ git rev-parse HEAD
9ce234ba236e00a1a7899622d7275fd6392b5b91
Exit 0; 0.0s; log: artifacts/local/tranche-1-demo/revision.log
$ git status --short
 M docs/local-rpc.md
?? notebooks/
Exit 0; 0.0s; log: artifacts/local/tranche-1-demo/worktree.log
$ rustc --version
rustc 1.84.1 (e71f9a9a9 2025-01-27)
Exit 0; 0.0s; log: artifacts/local/tranche-1-demo/rust.log
$ stellar --version
stellar 22.8.2 (7ab5f0565ed3a03b41789aaac7211c286fa0028c)
stellar-xdr 22.1.0 (e13922970800d95b523413018b2279df42df3442)
xdr curr (529d5176f24c73eeccfa5eba481d4e89c19b1181)
Exit 0; 0.0s; log: artifacts/local/tranche-1-demo/stellar.log
$ cargo llvm-cov --version
cargo-llvm-cov 0.6.16
Exit 0; 0.0s; log: artifacts/local/tranche-1-demo/coverage-tool.log


## P1 — complete build, tests and coverage (before recording)

This is the authoritative repository check: optimized upstream/Etesia builds, fmt, strict clippy, all native tests and a ≥90% production line-coverage assertion. Existing upstream DeFindex and LLVM macro warnings may appear; do not describe the entire dependency output as warning-free. The full log is saved.

In [2]:
check_output = run(["sh", "scripts/check.sh"], "full-check", tail=45)
CHECK_FINISHED_AT = datetime.now(timezone.utc).isoformat()
print("Native suite summaries:")
for line in check_output.splitlines():
    if "test result:" in line:
        print(line)

$ sh scripts/check.sh
test tests::recovery_consumes_only_the_missing_repayment_assets ... ok
test tests::recovery_rejects_deleveraging_that_worsens_pool_health ... ok
test tests::reward_claim_measures_custody_and_rejects_false_report_atomically ... ok
test tests::recovery_rejects_less_debt_when_losses_increase_leverage ... ok
test tests::stale_price_basket_waives_only_exiting_entitlement ... ok
test tests::reward_assets_cannot_be_bought_but_can_be_realized ... ok
test tests::transfers_allowances_expiration_and_delegated_exit ... ok
test tests::target_auth_guarded_swap_and_nonce_replay ... ok
test tests::price_validity_divergence_insolvency_and_liquidity_views ... ok
test tests::reject_expired_changed_or_unbounded_plans_before_custody_changes ... ok
test tests::recovery_allows_partial_deleveraging_while_limits_remain_breached ... ok
test tests::ttl_extension_and_restored_snapshot_preserve_supply_nonce_mark_and_pending_change ... ok
test tests::repeated_individually_legal_public_unwinds_

## P2 — start a fresh isolated node (before recording)

This starts a background process owned by the notebook, waits for readiness and uses a new data directory. Ports must be free. Run the cleanup cell when finished; do not restart the kernel before stopping the node. If the kernel is interrupted during startup, run cleanup. Setup can take several minutes.

In [3]:
for port in (8003, 11625, 11626, 11825, 11826, 1570):
    with socket.socket() as sock:
        assert sock.connect_ex(("127.0.0.1", port)) != 0, f"Port {port} is already in use; stop that node first"
(ROOT / ".tooling").mkdir(exist_ok=True)
NODE_DATA = Path(tempfile.mkdtemp(prefix="tranche-1-", dir=ROOT / ".tooling"))
with (OUT / "node.log").open("w") as stream:
    node = subprocess.Popen([sys.executable, "scripts/local-node.py", str(NODE_DATA)],
                            cwd=ROOT, env=ENV, stdout=stream, stderr=subprocess.STDOUT)
try:
    deadline = time.monotonic() + 240
    while "Local RPC ready" not in (OUT / "node.log").read_text():
        assert node.poll() is None, (OUT / "node.log").read_text()
        assert time.monotonic() < deadline, "Node startup timed out; inspect artifacts/local/tranche-1-demo/node.log"
        time.sleep(1)
    assert rpc("getNetwork")["passphrase"] == PASSPHRASE
    assert rpc("getHealth")["status"] == "healthy"
    print("Healthy isolated RPC:", RPC)
    print("Network:", PASSPHRASE)
except BaseException:
    node.terminate()
    node.wait(timeout=60)
    raise

Healthy isolated RPC: http://127.0.0.1:8003
Network: Standalone Network ; September 2026


## P3 — fresh D2 deployment and interaction (before recording)

Deploys six fixture assets and the vault, deposits 100 USDC, publishes a target, performs a guarded swap, collects fees, unwinds and redeems all holder shares. The existing smoke script asserts the holder's final balance is zero. This is the deployment segment to record separately if showing a time-compressed run.

In [4]:
run([sys.executable, "scripts/local-smoke.py"], "d2-local-deployment")
d2 = json.loads((ROOT / "artifacts/local/manifest.json").read_text())
assert d2["result"] == "passed" and d2["passphrase"] == PASSPHRASE
assert len(d2["assets"]) == 6
print("Deployed vault:", d2["vault"])
print("Deposit USDC:", Decimal(d2["deposited_assets"]) / 10**7)
print("Minted shares:", Decimal(d2["minted_shares"]) / 10**12)
print("Redeemed USDC:", Decimal(d2["redeemed_assets"]) / 10**7)

$ python3 scripts/local-smoke.py
funded USDC CA43JDCRC3BV4OOJGXRCDYVJRXGIQURIC7EHVHE2W37Q5JZWQ53VAIWT
funded XLM CBAWJP7YZ4YZNUFUROX2CSP37ZKKGGG4R4APNTPRZHXZCIM5PYM54VA5
funded AQUA CDM2HESHI2M6FTX4XVAAGCTMH5GOBDL4ZQUGCHA35P6FRAKQUVKALEQK
funded ETH CBMOV7XWW5EC6BYKBQ3L5T27QZ4G5XNWOEEHMWPVS3BXUQEGLM425PS7
funded BTC CACDAOOF7EECAFMAV3BUHZDI4OAOR5TVVCXP6WWXILEBAM3RKJZQC4C4
funded USTRY CCY62MGCD4O3QDZOXKROTPI6M3KUUJVJ55WYS4547LYI3OXNQV3SZMRJ
deposited 100000000031709
guarded swap confirmed
local RPC round trip passed 999999997
Exit 0; 40.3s; log: artifacts/local/tranche-1-demo/d2-local-deployment.log
Deployed vault: CB5ZA3DDQ2ODT2T4HAFPPKSAMCAOPXGUFVQDMX6K23L3XKJP6ME7B6UM
Deposit USDC: 100
Minted shares: 100.000000031709
Redeemed USDC: 99.9999997


## P4 — prepare the D3 pricing provider and local NAV round trip

The required Soroswap pair WASM is downloaded by the repository's unsigned source probe if absent. For a fresh checkout, first install the probe's locked dependencies with `corepack pnpm install --frozen-lockfile` from `tools/source-probe/`. The probe only reads mainnet; its historical SHX discovery does not add SHX to the current vault universe. Failure to obtain the pinned WASM stops this cell.

The local smoke deploys actual pair WASM with controlled local tokens/feed, collects three observations for each non-USDC asset, verifies 1,005 USDC NAV, deposits, rejects an invalid-feed redemption, restores the feed and redeems. Run before filming. Its fixture marks are not current mainnet quotes.

In [5]:
pair_hash = "18051456816b66f12e773a56f77c5794fac1b1fb7ab6e22d4fad5a412770f73e"
pair = ROOT / "artifacts/local/upstream" / (pair_hash + ".wasm")
if not pair.exists():
    run(["node", "tools/source-probe/probe.cjs", str(OUT / "source-probe.json")],
        "unsigned-source-probe", tail=8)
assert hashlib.sha256(pair.read_bytes()).hexdigest() == pair_hash
run([sys.executable, "scripts/d3-local-smoke.py"], "d3-local-deployment")
d3 = json.loads((ROOT / "artifacts/local/d3/manifest.json").read_text())
assert d3["result"] == "passed" and d3["nav"] == 10_050_000_000
print("Local NAV:", Decimal(d3["nav"]) / 10**7, "USDC")
print("Pricing provider:", d3["provider"])
print("Feed:", d3["feed"])

$ python3 scripts/d3-local-smoke.py
funded LOCAL USDC
funded LOCAL XLM
funded LOCAL AQUA
funded LOCAL ETH
funded LOCAL BTC
funded LOCAL USTRY
upstream pair ready XLM
upstream pair ready AQUA
upstream pair ready ETH
upstream pair ready BTC
upstream pair ready USTRY
collected round 1
collected round 2
collected round 3
D3 local RPC passed 10050000000 999999998
Exit 0; 100.4s; log: artifacts/local/tranche-1-demo/d3-local-deployment.log
Local NAV: 1005 USDC
Pricing provider: CCJAI5PQQS3SACHSXXBRMOFMMBSIAT5ST3D7TC2QWNWUMTXBH5ABBLWJ
Feed: controlled local ABI fixture; deployed Reflector v6 spec is incompatible with pinned CLI 22


## D2-A — warm compilation

Here I’m compiling the Etesia vault contracts into Soroban WebAssembly. The dependencies and full validation checks were prepared earlier, so this repeat build uses the existing cache. The output shows the optimized vault file, its size and its SHA-256 hash, which identifies the exact artifact used in the deployment.

In [6]:
run(["cargo", "build", "--workspace", "--locked", "--release",
     "--target", "wasm32-unknown-unknown"], "warm-build", tail=8)
run(["stellar", "contract", "optimize", "--wasm",
     "target/wasm32-unknown-unknown/release/etesia_vault.wasm"], "optimize-vault", tail=8)
wasm = ROOT / "target/wasm32-unknown-unknown/release/etesia_vault.optimized.wasm"
print(wasm.name, "bytes:", wasm.stat().st_size)
print("SHA-256:", hashlib.sha256(wasm.read_bytes()).hexdigest())

$ cargo build --workspace --locked --release --target wasm32-unknown-unknown
    Finished `release` profile [optimized] target(s) in 0.17s
Exit 0; 0.2s; log: artifacts/local/tranche-1-demo/warm-build.log
$ stellar contract optimize --wasm target/wasm32-unknown-unknown/release/etesia_vault.wasm
Reading: target/wasm32-unknown-unknown/release/etesia_vault.wasm (127274 bytes)
Optimized: target/wasm32-unknown-unknown/release/etesia_vault.optimized.wasm (98976 bytes)
Exit 0; 0.4s; log: artifacts/local/tranche-1-demo/optimize-vault.log
etesia_vault.optimized.wasm bytes: 98976
SHA-256: 605f3ce004d000a383aa88a2a3527acef82c7567713ee6932e6766054f28e568


## D2-B — measured coverage

This is the production line-coverage report from the full validation run. It covers the vault, adapter and pricing provider, with a required minimum of ninety percent. Tests, fixture tools, vendored dependencies and generated wire declarations are excluded. The displayed result records the measured coverage, test summaries, source revision and completion time.

In [7]:
lines = json.loads((ROOT / "coverage/coverage.json").read_text())["data"][0]["totals"]["lines"]
assert lines["percent"] >= 90
print("Full checks completed UTC:", CHECK_FINISHED_AT)
print("Source revision:", REVISION)
print(f"Production line coverage: {lines['covered']}/{lines['count']} = {lines['percent']:.2f}% (required: 90%)")
for line in check_output.splitlines():
    if "test result:" in line:
        print(line)

Full checks completed UTC: 2026-09-19T13:34:01.366394+00:00
Source revision: 9ce234ba236e00a1a7899622d7275fd6392b5b91
Production line coverage: 2204/2345 = 93.99% (required: 90%)
test result: ok. 3 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.55s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 16 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.17s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 44 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 1.72s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filte

## D2-C — share correctness and rejection of violations

I’m running four focused tests. They check deposit, mint, withdraw and redeem accounting, reject unauthorized share issuance, reject invalid rebalance plans and prevent nonce replay. Each command selects one exact test, so “one passed, forty-three filtered out” means that test passed and the other vault tests were not selected. The full suite ran during preparation.

In [8]:
test("etesia-vault", "funded_seed_and_all_standard_round_trips")
test("etesia-vault", "exact_deposit_auth_tree_and_no_unauthorized_issuance")
test("etesia-vault", "reject_expired_changed_or_unbounded_plans_before_custody_changes")
test("etesia-vault", "target_auth_guarded_swap_and_nonce_replay")

$ cargo test --locked -p etesia-vault tests::funded_seed_and_all_standard_round_trips -- --exact --nocapture
    Finished `test` profile [unoptimized] target(s) in 0.10s
     Running unittests src/lib.rs (target/debug/deps/etesia_vault-1283b7f722122507)

running 1 test
Writing test snapshot file for test "tests::funded_seed_and_all_standard_round_trips" to "test_snapshots/tests/funded_seed_and_all_standard_round_trips.1.json".
test tests::funded_seed_and_all_standard_round_trips ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 43 filtered out; finished in 0.13s

Exit 0; 0.3s; log: artifacts/local/tranche-1-demo/funded_seed_and_all_standard_round_trips.log
$ cargo test --locked -p etesia-vault tests::exact_deposit_auth_tree_and_no_unauthorized_issuance -- --exact --nocapture
    Finished `test` profile [unoptimized] target(s) in 0.09s
     Running unittests src/lib.rs (target/debug/deps/etesia_vault-1283b7f722122507)

running 1 test
Writing test snapshot file for test

## D2-D — re-query local transaction receipts

The preparation run deployed the vault, deposited one hundred USDC, issued shares, executed a guarded action and redeemed the holder’s position. I’m now querying the local RPC to verify the recorded D2 and D3 transaction receipts and checking that the WASM hashes match our build artifacts. These transactions use test assets on an isolated local network.

In [9]:
assert rpc("getNetwork")["passphrase"] == PASSPHRASE
print("Isolated network:", PASSPHRASE)
verified = {}
for label, manifest in [("D2", d2), ("D3", d3)]:
    assert manifest["transactions"], "No transaction receipts captured"
    receipts = []
    for item in manifest["transactions"]:
        receipt = rpc("getTransaction", hash=item["hash"])
        assert receipt["status"] == "SUCCESS", receipt
        receipts.append({**item, "status": receipt["status"], "ledger": receipt["ledger"]})
    for name, digest in manifest["wasm_sha256"].items():
        artifact = ROOT / "target/wasm32-unknown-unknown/release" / name
        assert hashlib.sha256(artifact.read_bytes()).hexdigest() == digest, name
    verified[label] = {"vault": manifest["vault"], "receipts": receipts,
                       "wasm_sha256": manifest["wasm_sha256"]}
    print(label, manifest["vault"], len(receipts), "SUCCESS receipts; WASM hashes match")
    for item in receipts:
        if item["method"] in ("deposit", "execute", "redeem"):
            print(item["method"], item["status"], "ledger", item["ledger"], item["hash"])
(OUT / "verified-receipts.json").write_text(json.dumps({
    "verified_at": datetime.now(timezone.utc).isoformat(), "revision": REVISION,
    "network": PASSPHRASE, "scope": "local fixture transactions only", "scenarios": verified
}, indent=2) + "\n")

Isolated network: Standalone Network ; September 2026
D2 CB5ZA3DDQ2ODT2T4HAFPPKSAMCAOPXGUFVQDMX6K23L3XKJP6ME7B6UM 26 SUCCESS receipts; WASM hashes match
deposit SUCCESS ledger 87 560c1c43e1d2b14a7dfbdf18b652217a3c1fd8d1d2e58d187353b4fa627884ae
execute SUCCESS ledger 91 ffe843ce85715878a4699871db0aab6ffdf56ff0a6b247285a15daeb7a9a6b27
redeem SUCCESS ledger 96 0b801a74ca9eb8bc66676b5ad6dac12129e602ce0a5c3c55d50b2b895d73f206
D3 CB7W3QPI47Q36EYOOAO572GYVJEYELLMBEEEMBB3HL3YAM7DDYNDYMJF 69 SUCCESS receipts; WASM hashes match
deposit SUCCESS ledger 318 0ea9fe0faec31b7eef38833937fa34efb5b3e7cafbb658448dd1b794c92b28b7
redeem SUCCESS ledger 323 6081d9c57369bc802785561d4aa276c54c9037a10350f5a20120dab74d3fd54f


19859

## D3-A — read both price values and calculate deviation

This demonstration uses real Soroswap pool code with local test liquidity and a controlled Reflector-interface feed. The provider reads the same pool at three different times and calculates a TWAP. It compares that value with the feed price, with both expressed in USDC per asset. 

In [10]:
asset = next(a["address"] for a in d3["assets"] if a["symbol"] == "XLM")
source = next(s for s in d3["provider_config"]["sources"] if s["asset"] == asset)
feed = source["numerator"]["contract"]
provider = d3["provider"]
assert invoke(provider, "config_hash") == d3["config_hash"]

def set_fixture_price(price):
    invoke(feed, "update", ["--asset", json.dumps({"Stellar": asset}),
                           "--price", str(price), "--timestamp", str(int(time.time()))], send="yes")

set_fixture_price(10**14)
for _ in range(3):
    observation = invoke(provider, "observe", ["--asset", asset], send="yes")
    deadline = time.monotonic() + 20
    while rpc("getLatestLedger")["sequence"] <= observation["ledger"]:
        assert time.monotonic() < deadline, "Wait for a new local ledger before retrying"
        time.sleep(1)
diagnostic = invoke(provider, "diagnose", ["--asset", asset])
assert diagnostic["error"] == 0 and diagnostic["mark"]["ready"], diagnostic
samples = diagnostic["samples"]
(p0, p1, p2) = [int(s["price"]) for s in samples]
(t0, t1, t2) = [s["timestamp"] for s in samples]
twap = ((p0+p1)*(t1-t0) + (p1+p2)*(t2-t1)) // (2*(t2-t0))
mark = invoke(provider, "mark", ["--asset", asset])
reflector = int(mark["price"])
assert twap == int(mark["reference"])
deviation = (abs(twap-reflector)*10000 + reflector-1) // reflector
print("LOCAL controlled feed; actual pair WASM; XLM fixture")
print("Observations (timestamp, price):", [(s["timestamp"], s["price"]) for s in samples])
print("Reflector-interface mark:", Decimal(reflector)/10**12, "USDC/asset")
print("Soroswap TWAP:", Decimal(twap)/10**12, "USDC/asset")
print("Deviation:", deviation, "bps; configured limit:", source["divergence_bps"], "bps")
assert deviation <= source["divergence_bps"]

LOCAL controlled feed; actual pair WASM; XLM fixture
Observations (timestamp, price): [(1789825290, '1000000000000'), (1789825292, '1000000000000'), (1789825294, '1000000000000')]
Reflector-interface mark: 1 USDC/asset
Soroswap TWAP: 1 USDC/asset
Deviation: 0 bps; configured limit: 100 bps


## D3-B — breach the deviation limit and restore agreement

I’m raising the controlled feed price to one point zero two USDC while the pool price stays at one. The difference, divided by the Reflector mark and rounded up, is one hundred ninety-seven basis points. That exceeds the configured one-hundred-basis-point limit, so the provider rejects the price with a divergence error. Restoring agreement makes a validated price available again.

In [11]:
try:
    set_fixture_price(102_000_000_000_000)
    rejected = invoke(provider, "diagnose", ["--asset", asset])
    assert rejected["error"] == 13 and not rejected["mark"]["ready"], rejected
    price, reference = int(rejected["mark"]["price"]), int(rejected["mark"]["reference"])
    bps = (abs(price-reference)*10000 + price-1) // price
    print("Deviation:", bps, "bps; limit:", source["divergence_bps"], "bps")
    print("Diagnostic error:", rejected["error"], "Divergence; ready:", rejected["mark"]["ready"])
    try:
        invoke(provider, "mark", ["--asset", asset])
    except RuntimeError as error:
        assert "#13" in str(error), str(error)
        print("mark rejected with contract error #13")
    else:
        raise AssertionError("Divergent price was accepted")
finally:
    set_fixture_price(10**14)
assert invoke(provider, "mark", ["--asset", asset])["ready"]
print("Agreement restored; validated mark available again")

Deviation: 197 bps; limit: 100 bps
Diagnostic error: 13 Divergence; ready: False
mark rejected with contract error #13
Agreement restored; validated mark available again


## D3-C — NAV, shares and atomic refusal

This contract test checks a one-thousand-USDC seed, a one-USDC deposit and a donated asset worth one hundred USDC, giving a NAV of one thousand one hundred and one USDC. It also verifies that unavailable pricing blocks deposits and redemption without changing state, holdings, shares or events. Valid pricing allows redemption; stale pricing blocks valuation. This is a controlled simulation, and full-universe live pricing qualification remains open.

In [12]:
test("etesia-pricing", "new_vault_uses_provider_and_failures_preserve_state")
print("Native contract assertion: 1,000 seed + 1 deposit + 100 asset value = 1,101 USDC NAV")
print("Separate local RPC assertion from P4:", Decimal(d3["nav"])/10**7, "USDC")

$ cargo test --locked -p etesia-pricing tests::new_vault_uses_provider_and_failures_preserve_state -- --exact --nocapture
   Compiling etesia-vault v0.1.0 (./contracts/vault)
   Compiling etesia-pricing v0.1.0 (./contracts/pricing)
    Finished `test` profile [unoptimized] target(s) in 5.28s
     Running unittests src/lib.rs (target/debug/deps/etesia_pricing-42e9f6379fc4a01d)

running 1 test
Writing test snapshot file for test "tests::new_vault_uses_provider_and_failures_preserve_state" to "test_snapshots/tests/new_vault_uses_provider_and_failures_preserve_state.1.json".
test tests::new_vault_uses_provider_and_failures_preserve_state ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 15 filtered out; finished in 0.14s

Exit 0; 5.5s; log: artifacts/local/tranche-1-demo/new_vault_uses_provider_and_failures_preserve_state.log
Native contract assertion: 1,000 seed + 1 deposit + 100 asset value = 1,101 USDC NAV
Separate local RPC assertion from P4: 1005 USDC


## Cleanup — after recording

Stop only the process started by this notebook. Keep logs and node data for review. Public submission still needs the uploaded video, a public source revision and resolution/disclosure of the acceptance gaps in the repository's D1/D3 records.

In [13]:
if "node" in globals() and node.poll() is None:
    node.terminate()
    node.wait(timeout=60)
    print("Notebook-owned local node stopped; artifacts retained")

Notebook-owned local node stopped; artifacts retained
